# Case Study 2: Sentiment Classification

Grouped five-fold comparison, calibrated probabilities, and selective triage that defers uncertain reviews to an analyst.

In [ ]:
import glob, json, os, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

PACKAGES = ["gensim"]
KAGGLE = os.path.isdir("/kaggle/working")
if KAGGLE:
    if PACKAGES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)
    hits = glob.glob("/kaggle/input/**/telelens/__init__.py", recursive=True)
    if hits:
        ROOT = Path(hits[0]).parents[2]
    else:
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/BharatChandra-sys/NLP-case-study.git", "/tmp/repo"], check=True)
        ROOT = Path("/tmp/repo")
    RESULTS = Path("/kaggle/working/results/sentiment")
else:
    ROOT = Path.cwd().resolve().parent
    RESULTS = ROOT / "results" / "sentiment"
sys.path.insert(0, str(ROOT / "src"))
RESULTS.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print("device", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")


def save_json(name, payload):
    (RESULTS / name).write_text(json.dumps(payload, indent=2, default=float), encoding="utf-8")


In [ ]:
import re
from gensim.models import FastText
from sklearn.metrics import confusion_matrix, f1_score
from telelens.data import load_comcast, load_telecom
from telelens.embeddings import Embeddings, log_odds_polarity, polarity_counterfit, train_sgns
from telelens.metrics import (bootstrap_ci, bootstrap_macro_f1_diff, classification_report, expected_calibration_error,
                              mcnemar_exact, risk_coverage, threshold_for_risk)
from telelens.sentiment import dense_lr, distilbert_finetune, grouped_folds, hybrid_lr, pretrained_sst2, tfidf_lr
from telelens.text import bind_negation, tokenize
from telelens import viz

frame = load_telecom(10000)
small = load_telecom(1000)
y = frame["label"].to_numpy()
texts = frame["text"].tolist()
plain = [tokenize(t) for t in texts]
bound = [bind_negation(d) for d in plain]
folds = grouped_folds(frame)
print(len(frame), np.bincount(y))

## Continuity with the earlier report: TF-IDF on the 1K file

In [ ]:
oof = np.zeros(len(small))
for tr, te in grouped_folds(small):
    oof[te] = tfidf_lr(small["text"].iloc[tr], small["label"].iloc[tr], small["text"].iloc[te])
continuity = classification_report(small["label"], (oof >= 0.5).astype(int))
save_json("continuity_1k.json", continuity)
continuity

## Five-fold comparison on the 10K file

In [ ]:
names = ["TF-IDF LR", "SGNS mean LR", "FastText mean LR", "SGNS+neg mean LR", "PA-W2V mean LR", "TF-IDF + SGNS", "TF-IDF + PA-W2V", "DistilBERT fine-tuned"]
probs = {n: np.zeros(len(frame)) for n in names}
fold_of = np.zeros(len(frame), dtype=int)
temps = []
timing = {n: 0.0 for n in names}


def timed(name, fn):
    start = time.time()
    value = fn()
    timing[name] += time.time() - start
    return value


for fold, (tr, te) in enumerate(folds):
    fold_of[te] = fold
    x_tr, x_te = [texts[i] for i in tr], [texts[i] for i in te]
    probs["TF-IDF LR"][te] = timed("TF-IDF LR", lambda: tfidf_lr(x_tr, y[tr], x_te))
    sgns, _ = train_sgns([plain[i] for i in tr], device=DEVICE)
    a, b = sgns.embed_many([plain[i] for i in tr]), sgns.embed_many([plain[i] for i in te])
    probs["SGNS mean LR"][te] = timed("SGNS mean LR", lambda: dense_lr(a, y[tr], b))
    probs["TF-IDF + SGNS"][te] = hybrid_lr(x_tr, a, y[tr], x_te, b)
    fast = FastText([plain[i] for i in tr], vector_size=200, window=5, min_count=5, sg=1, negative=10, epochs=5, seed=SEED, workers=4, min_n=3, max_n=6)
    emb = Embeddings(fast.wv.index_to_key, fast.wv.vectors.copy())
    probs["FastText mean LR"][te] = dense_lr(emb.embed_many([plain[i] for i in tr]), y[tr], emb.embed_many([plain[i] for i in te]))
    base, _ = train_sgns([bound[i] for i in tr], device=DEVICE, bound=True)
    a, b = base.embed_many([bound[i] for i in tr]), base.embed_many([bound[i] for i in te])
    probs["SGNS+neg mean LR"][te] = dense_lr(a, y[tr], b)
    pa, _ = polarity_counterfit(base, log_odds_polarity([bound[i] for i in tr], y[tr]), device=DEVICE)
    a, b = pa.embed_many([bound[i] for i in tr]), pa.embed_many([bound[i] for i in te])
    probs["PA-W2V mean LR"][te] = dense_lr(a, y[tr], b)
    probs["TF-IDF + PA-W2V"][te] = hybrid_lr(x_tr, a, y[tr], x_te, b)
    p, temp = timed("DistilBERT fine-tuned", lambda: distilbert_finetune(x_tr, y[tr], x_te, DEVICE))
    probs["DistilBERT fine-tuned"][te] = p
    temps.append(temp)
    print("fold", fold, {k: round(f1_score(y[te], probs[k][te] >= 0.5, average="macro"), 4) for k in names})
probs["DistilBERT SST-2 (no training)"] = pretrained_sst2(texts, DEVICE)
pd.DataFrame({"label": y, "fold": fold_of, **probs}).to_csv(RESULTS / "oof_probabilities.csv", index_label="row")

In [ ]:
rows = []
for name, p in probs.items():
    pred = (p >= 0.5).astype(int)
    fold_f1 = [f1_score(y[te], pred[te], average="macro") for _, te in folds]
    lo, hi = bootstrap_ci((pred == y).astype(float))
    rows.append({"model": name, **classification_report(y, pred), "fold_f1_std": np.std(fold_f1), "acc_lo": lo, "acc_hi": hi,
                 "ece": expected_calibration_error(y, p)})
scores = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
scores.to_csv(RESULTS / "scores.csv", index=False)
comparisons = []
for a, b in [("TF-IDF + PA-W2V", "TF-IDF LR"), ("TF-IDF + PA-W2V", "TF-IDF + SGNS"), ("PA-W2V mean LR", "SGNS mean LR"), ("PA-W2V mean LR", "SGNS+neg mean LR"),
             ("SGNS+neg mean LR", "SGNS mean LR"), ("DistilBERT fine-tuned", "TF-IDF LR"), ("DistilBERT fine-tuned", "TF-IDF + PA-W2V")]:
    pa, pb = (probs[a] >= 0.5).astype(int), (probs[b] >= 0.5).astype(int)
    mean, lo, hi = bootstrap_macro_f1_diff(y, pa, pb)
    a_only, b_only, p_value = mcnemar_exact(y, pa, pb)
    comparisons.append({"a": a, "b": b, "f1_diff": mean, "ci_lo": lo, "ci_hi": hi, "a_only_correct": a_only, "b_only_correct": b_only, "mcnemar_p": p_value})
comparisons = pd.DataFrame(comparisons)
comparisons.to_csv(RESULTS / "comparisons.csv", index=False)
save_json("timing_seconds.json", timing)
save_json("temperatures.json", temps)
display(scores.round(4), comparisons.round(4))

## Selective triage: deferral thresholds are cross-fitted across folds

In [ ]:
TARGETS = [0.10, 0.05, 0.03, 0.02]
selective = []
for name in ["TF-IDF LR", "TF-IDF + PA-W2V", "DistilBERT fine-tuned"]:
    p = probs[name]
    for target in TARGETS:
        kept = np.zeros(len(y), dtype=bool)
        for k in range(5):
            tau = threshold_for_risk(y[fold_of != k], p[fold_of != k], target)
            kept[fold_of == k] = np.maximum(p[fold_of == k], 1 - p[fold_of == k]) >= tau
        pred = (p >= 0.5).astype(int)
        bad_auto = (y == 0) & kept
        selective.append({"model": name, "target_risk": target, "coverage": kept.mean(), "risk": 1 - (pred[kept] == y[kept]).mean(),
                          "deferred": int((~kept).sum()), "auto_bad_recall": (pred[bad_auto] == 0).mean(),
                          "missed_bad_auto": int(((pred == 1) & bad_auto).sum())})
selective = pd.DataFrame(selective)
selective.to_csv(RESULTS / "selective.csv", index=False)
curves = []
for name in ["TF-IDF LR", "TF-IDF + PA-W2V", "DistilBERT fine-tuned"]:
    rc = risk_coverage(y, probs[name])
    curves.append((name, [r["coverage"] for r in rc], [r["risk"] for r in rc]))
viz.lines(RESULTS / "risk_coverage.png", curves, "Risk falls as more reviews are deferred", "Coverage (share handled automatically)", "Error rate on handled reviews", xlim=(0.2, 1.0))
ordered = scores.sort_values("macro_f1", ascending=False)
viz.hbar(RESULTS / "model_f1.png", ordered["model"].tolist(), ordered["macro_f1"].tolist(), "Out-of-fold macro-F1, 10K grouped CV", "Macro-F1", xlim=(0.6, 1.0))
best = (probs["DistilBERT fine-tuned"] >= 0.5).astype(int)
save_json("confusion_distilbert.json", confusion_matrix(y, best).tolist())
selective.round(4)

## Robustness: contrast set and cross-domain test on Comcast complaints

In [ ]:
CONTRAST = [("the service is reliable", 1), ("the service is not reliable", 0), ("the connection is stable", 1), ("the connection is unstable", 0),
            ("the engineer was helpful", 1), ("the engineer was not helpful", 0), ("my problem was fixed quickly", 1), ("my problem was never fixed", 0),
            ("i would recommend this provider", 1), ("i would not recommend this provider", 0), ("the staff were polite", 1), ("the staff were not polite", 0),
            ("the broadband is fast", 1), ("the broadband is not fast", 0), ("billing is clear and easy", 1), ("billing is not clear or easy", 0),
            ("no problems at all with the router", 1), ("nothing but problems with the router", 0), ("i am happy with the price", 1), ("i am not happy with the price", 0),
            ("she was very friendly", 1), ("he was very unfriendly", 0), ("installation was smooth", 1), ("installation was a nightmare", 0),
            ("never had an issue with the signal", 1), ("always have an issue with the signal", 0), ("not bad at all", 1), ("not good at all", 0),
            ("customer support could not be better", 1), ("customer support could not be worse", 0)]
contrast_x, contrast_y = [c[0] for c in CONTRAST], np.array([c[1] for c in CONTRAST])
comcast = load_comcast()
comcast = comcast[comcast["rating"].isin([1, 4, 5])]
clean = [" ".join(re.findall(r"[a-z]+", t.lower().replace("'", ""))) for t in comcast["text"]]
comcast_y = (comcast["rating"] >= 4).astype(int).to_numpy()
base, _ = train_sgns(bound, device=DEVICE, bound=True)
pa, _ = polarity_counterfit(base, log_odds_polarity(bound, y), device=DEVICE)
train_x = pa.embed_many(bound)
embed = lambda items: pa.embed_many([bind_negation(tokenize(t)) for t in items])
targets = contrast_x + clean
outputs = {"TF-IDF LR": tfidf_lr(texts, y, targets), "TF-IDF + PA-W2V": hybrid_lr(texts, train_x, y, targets, embed(targets)),
           "PA-W2V mean LR": dense_lr(train_x, y, embed(targets)), "DistilBERT fine-tuned": distilbert_finetune(texts, y, targets, DEVICE)[0]}
n = len(contrast_x)
robust = []
contrast_detail = pd.DataFrame({"text": contrast_x, "label": contrast_y})
for name, p in outputs.items():
    contrast_detail[name] = p[:n].round(3)
    robust.append({"model": name, "contrast_accuracy": ((p[:n] >= 0.5) == contrast_y).mean(), "comcast_macro_f1": f1_score(comcast_y, p[n:] >= 0.5, average="macro"),
                   **{f"comcast_{k}": v for k, v in classification_report(comcast_y, (p[n:] >= 0.5).astype(int)).items() if k in ("accuracy", "bad_recall")},
                   "comcast_good_n": int(comcast_y.sum()), "comcast_bad_n": int((1 - comcast_y).sum())})
robust = pd.DataFrame(robust)
robust.to_csv(RESULTS / "robustness.csv", index=False)
contrast_detail.to_csv(RESULTS / "contrast_detail.csv", index=False)
robust.round(4)

## Recorded Kaggle results

Executed on a Kaggle T4 GPU as `telelens-02-sentiment`. Files are in `results/sentiment/`.

**scores.csv**

| model | accuracy | macro_f1 | bad_precision | bad_recall | bad_f1 | fold_f1_std | acc_lo | acc_hi | ece |
|---|---|---|---|---|---|---|---|---|---|
| DistilBERT fine-tuned | 0.9158 | 0.9158 | 0.9219 | 0.9099 | 0.9159 | 0.0086 | 0.9107 | 0.9214 | 0.0241 |
| TF-IDF LR | 0.8969 | 0.8968 | 0.8809 | 0.9195 | 0.8998 | 0.0061 | 0.8909 | 0.9027 | 0.0129 |
| TF-IDF + PA-W2V | 0.8956 | 0.8955 | 0.8771 | 0.9218 | 0.8989 | 0.0093 | 0.8895 | 0.9018 | 0.0192 |
| TF-IDF + SGNS | 0.8907 | 0.8906 | 0.8721 | 0.9175 | 0.8942 | 0.0122 | 0.8848 | 0.8969 | 0.0236 |
| PA-W2V mean LR | 0.8855 | 0.8853 | 0.8655 | 0.9147 | 0.8894 | 0.0034 | 0.8792 | 0.8915 | 0.0323 |
| SGNS+neg mean LR | 0.8779 | 0.8777 | 0.8607 | 0.9037 | 0.8817 | 0.0106 | 0.8715 | 0.8839 | 0.0284 |
| SGNS mean LR | 0.8715 | 0.8713 | 0.8517 | 0.9019 | 0.876 | 0.0102 | 0.865 | 0.878 | 0.0306 |
| FastText mean LR | 0.8631 | 0.8628 | 0.84 | 0.8994 | 0.8687 | 0.0088 | 0.8565 | 0.8696 | 0.0328 |
| DistilBERT SST-2 (no training) | 0.8231 | 0.8187 | 0.7506 | 0.9714 | 0.8468 | 0.0122 | 0.815 | 0.8312 | 0.159 |

**comparisons.csv**

| a | b | f1_diff | ci_lo | ci_hi | a_only_correct | b_only_correct | mcnemar_p |
|---|---|---|---|---|---|---|---|
| TF-IDF + PA-W2V | TF-IDF LR | -0.0012 | -0.0057 | 0.0032 | 253 | 266 | 0.5984 |
| TF-IDF + PA-W2V | TF-IDF + SGNS | 0.0049 | 0.0011 | 0.0088 | 223 | 175 | 0.0184 |
| PA-W2V mean LR | SGNS mean LR | 0.014 | 0.0087 | 0.0193 | 419 | 281 | 0.0 |
| PA-W2V mean LR | SGNS+neg mean LR | 0.0076 | 0.0029 | 0.0119 | 278 | 203 | 0.0007 |
| SGNS+neg mean LR | SGNS mean LR | 0.0064 | 0.0022 | 0.0107 | 289 | 226 | 0.0062 |
| DistilBERT fine-tuned | TF-IDF LR | 0.019 | 0.0132 | 0.0249 | 526 | 339 | 0.0 |
| DistilBERT fine-tuned | TF-IDF + PA-W2V | 0.0202 | 0.0144 | 0.026 | 548 | 348 | 0.0 |

**selective.csv**

| model | target_risk | coverage | risk | deferred | auto_bad_recall | missed_bad_auto |
|---|---|---|---|---|---|---|
| TF-IDF LR | 0.1 | 0.9887 | 0.0992 | 112 | 0.9232 | 378 |
| TF-IDF LR | 0.05 | 0.8412 | 0.0486 | 1568 | 0.9658 | 145 |
| TF-IDF LR | 0.03 | 0.7115 | 0.0295 | 2849 | 0.9796 | 73 |
| TF-IDF LR | 0.02 | 0.6133 | 0.0196 | 3818 | 0.9851 | 45 |
| TF-IDF + PA-W2V | 0.1 | 0.9873 | 0.0994 | 125 | 0.9264 | 362 |
| TF-IDF + PA-W2V | 0.05 | 0.8211 | 0.0507 | 1766 | 0.9625 | 157 |
| TF-IDF + PA-W2V | 0.03 | 0.6759 | 0.03 | 3200 | 0.9773 | 77 |
| TF-IDF + PA-W2V | 0.02 | 0.5336 | 0.0192 | 4605 | 0.9808 | 48 |
| DistilBERT fine-tuned | 0.1 | 1.0 | 0.0842 | 0 | 0.9099 | 448 |
| DistilBERT fine-tuned | 0.05 | 0.9097 | 0.0495 | 892 | 0.9465 | 242 |
| DistilBERT fine-tuned | 0.03 | 0.8168 | 0.0316 | 1809 | 0.9671 | 133 |
| DistilBERT fine-tuned | 0.02 | 0.7166 | 0.0194 | 2798 | 0.9822 | 63 |

**robustness.csv**

| model | contrast_accuracy | comcast_macro_f1 | comcast_accuracy | comcast_bad_recall | comcast_good_n | comcast_bad_n |
|---|---|---|---|---|---|---|
| TF-IDF LR | 0.6667 | 0.6851 | 0.9852 | 0.9941 | 51 | 3734 |
| TF-IDF + PA-W2V | 0.7333 | 0.6749 | 0.9839 | 0.9928 | 51 | 3734 |
| PA-W2V mean LR | 0.7333 | 0.6411 | 0.9783 | 0.9871 | 51 | 3734 |
| DistilBERT fine-tuned | 0.8667 | 0.6987 | 0.9815 | 0.9885 | 51 | 3734 |

![model_f1.png](../results/sentiment/model_f1.png)

![risk_coverage.png](../results/sentiment/risk_coverage.png)

